# DaTSCAN — Fase 3: ROI bilateral emparejada y Elastic Net (v3)

Este notebook localiza automáticamente la región estriatal en cada volumen preprocesado, audita la localización por protocolo, extrae características bilaterales y compara el mismo Elastic Net mediante:

1. validación estratificada convencional;
2. validación agrupada por protocolos latentes.

El clúster, el fold y las coordenadas de localización se conservan para auditoría, pero **no se utilizan como predictores**.

## 0. Dependencias

In [ ]:
# Descomente solo si falta algún paquete:
# %pip install numpy pandas scipy scikit-learn matplotlib seaborn joblib

## 1. Librerías y configuración

In [ ]:
from pathlib import Path
import os
import time, warnings
import joblib
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.ndimage import gaussian_filter, gaussian_filter1d
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 150)
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
import sys
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks': REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path: sys.path.insert(0, str(REPO_ROOT))
from src.config import DATA_ROOT
PROJECT_DIR = DATA_ROOT / 'latent_protocol_cv'
PREPROCESS_DIR = PROJECT_DIR / 'preprocessed_96x96x64_v2'
FOLDS_CSV = PROJECT_DIR / 'outputs' / 'train_protocol_folds.csv'
# Carpeta independiente: no reutiliza características de las versiones anteriores.
FEATURE_DIR = PROJECT_DIR / 'roi_adaptive_v3'
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 20260910
OUTPUT_SHAPE = (96,96,64)
SEARCH_X = (24,72)
SEARCH_Y = (20,76)
SEARCH_Z = (8,56)
MIDLINE_CANDIDATES = range(44,53)
PAIR_HALF_DISTANCES = range(7,18)
SAMPLES_PER_CLUSTER = 2
ROI_RADII = (3,5,7)
SLAB_HALF_WIDTHS = (1,3,5)

print('Volúmenes:', PREPROCESS_DIR, '| existe:', PREPROCESS_DIR.exists())
print('Folds:', FOLDS_CSV, '| existe:', FOLDS_CSV.exists())
if not PREPROCESS_DIR.exists() or not FOLDS_CSV.exists():
    raise FileNotFoundError('Falta la salida de una fase anterior.')

## 2. Manifiesto y verificación de los 1,362 volúmenes

In [ ]:
manifest = pd.read_csv(FOLDS_CSV)
manifest['processed_path'] = manifest['uid'].map(lambda uid: str((PREPROCESS_DIR/f'{uid}.npz').resolve()))
missing = manifest.loc[~manifest['processed_path'].map(lambda p: Path(p).exists())]
if len(missing):
    display(missing.head())
    raise FileNotFoundError(f'Faltan {len(missing)} volúmenes procesados.')
if manifest['uid'].duplicated().any():
    raise ValueError('Hay UID duplicados.')
if set(manifest['target'].unique()) != {0,1}:
    raise ValueError('La variable objetivo no es binaria 0/1.')
print('Estudios listos:',len(manifest))
display(pd.crosstab(manifest['protocol_cluster'],manifest['target'],margins=True))

## 3. Localización estriatal sin utilizar la etiqueta
Primero se estima el plano medio mediante simetría de baja frecuencia. Después se busca una pareja de ROI reflejadas respecto a ese plano. La puntuación permite que un lado sea débil, evitando desplazar la ROI patológica hacia el lado conservado.

In [ ]:
def load_volume(path):
    with np.load(path) as f:
        volume = f['volume'].astype(np.float32)
    if volume.shape != OUTPUT_SHAPE or not np.isfinite(volume).all():
        raise ValueError(f'Volumen inválido: {path} {volume.shape}')
    return volume

def estimate_midline(volume):
    anatomy=gaussian_filter(volume,sigma=(4.0,4.0,2.0))
    scores={}
    width=20
    for middle in MIDLINE_CANDIDATES:
        left=anatomy[middle-width:middle,20:76,12:52]
        right=anatomy[middle:middle+width,20:76,12:52][::-1]
        scale=float(np.mean(np.abs(left))+np.mean(np.abs(right))+1e-6)
        scores[middle]=float(np.mean(np.abs(left-right))/scale)
    middle=min(scores,key=scores.get)
    return int(middle),scores

def paired_candidate(smooth,z,middle):
    x0,x1=SEARCH_X;y0,y1=SEARCH_Y
    slab=smooth[:,:,max(0,z-1):min(smooth.shape[2],z+2)].mean(axis=2)
    baseline_l=float(np.percentile(slab[x0:middle,y0:y1],60))
    baseline_r=float(np.percentile(slab[middle:x1,y0:y1],60))
    best=None
    for half_distance in PAIR_HALF_DISTANCES:
        lx,rx=middle-half_distance,middle+half_distance
        if lx<x0 or rx>=x1:continue
        for y in range(y0+2,y1-2):
            contrast_l=max(float(slab[lx,y]-baseline_l),0.0)
            contrast_r=max(float(slab[rx,y]-baseline_r),0.0)
            # El foco fuerte guía la posición; el débil todavía contribuye.
            signal=max(contrast_l,contrast_r)+0.35*min(contrast_l,contrast_r)
            separation_penalty=np.exp(-0.5*((2*half_distance-18.0)/8.0)**2)
            y_position=(y-(y0+2))/max(1,(y1-3)-(y0+2))
            y_penalty=max(np.sin(np.pi*y_position),0.10)**0.35
            z0,z1=SEARCH_Z
            z_position=(z-z0)/max(1,z1-z0-1)
            z_penalty=max(np.sin(np.pi*z_position),0.05)**0.5
            score=signal*separation_penalty*y_penalty*z_penalty
            candidate={'score':float(score),'left':(lx,y,z),'right':(rx,y,z),
                       'half_distance':int(half_distance),'contrast_l':contrast_l,'contrast_r':contrast_r}
            if best is None or candidate['score']>best['score']:best=candidate
    if best is None:raise ValueError('No se pudo formar una pareja bilateral.')
    return best

def locate_striatum(volume):
    smooth=gaussian_filter(volume,sigma=(1.4,1.4,0.8))
    middle,midline_scores=estimate_midline(volume)
    z0,z1=SEARCH_Z
    candidates=[paired_candidate(smooth,z,middle) for z in range(z0,z1)]
    raw_profile=np.array([candidate['score'] for candidate in candidates])
    z_profile=gaussian_filter1d(raw_profile,sigma=1.0)
    best_index=int(np.argmax(z_profile))
    z_peak=int(z0+best_index)
    best=candidates[best_index]
    left=best['left'];right=best['right']
    slab=smooth[:,:,max(0,z_peak-2):min(volume.shape[2],z_peak+3)].mean(axis=2)
    boundary_distance=int(min(z_peak-z0,(z1-1)-z_peak))
    y_boundary_distance=int(min(left[1]-(SEARCH_Y[0]+2),(SEARCH_Y[1]-3)-left[1]))
    prominence=float(z_profile[best_index]/(np.median(z_profile)+1e-6))
    distance=float(2*best['half_distance'])
    warning=bool(boundary_distance<3 or y_boundary_distance<2 or best['half_distance'] in (min(PAIR_HALF_DISTANCES),max(PAIR_HALF_DISTANCES)) or prominence<1.08 or middle in (min(MIDLINE_CANDIDATES),max(MIDLINE_CANDIDATES)))
    return {'z_peak':z_peak,'left_x':left[0],'left_y':left[1],'right_x':right[0],'right_y':right[1],
            'midline_x':middle,'pair_half_distance':best['half_distance'],
            'peak_distance':distance,'peak_y_difference':0.0,
            'left_contrast':best['contrast_l'],'right_contrast':best['contrast_r'],
            'bilateral_score':float(z_profile[best_index]),'score_prominence':prominence,
            'boundary_distance':boundary_distance,'y_boundary_distance':y_boundary_distance,
            'localization_warning':warning,
            'z_profile':z_profile,'left':left,'right':right,'slab':slab}

def sphere_values(volume, center, radius, slab_half_width=None):
    cx,cy,cz=center
    z_radius=radius if slab_half_width is None else min(radius,slab_half_width)
    xlo,xhi=max(0,cx-radius),min(volume.shape[0],cx+radius+1)
    ylo,yhi=max(0,cy-radius),min(volume.shape[1],cy+radius+1)
    zlo,zhi=max(0,cz-z_radius),min(volume.shape[2],cz+z_radius+1)
    xx,yy,zz=np.ogrid[xlo:xhi,ylo:yhi,zlo:zhi]
    mask=(xx-cx)**2+(yy-cy)**2+(zz-cz)**2<=radius**2
    if slab_half_width is not None:mask &= np.abs(zz-cz)<=slab_half_width
    return volume[xlo:xhi,ylo:yhi,zlo:zhi][mask]

def summarize_values(values,prefix):
    return {f'{prefix}_mean':float(values.mean()),f'{prefix}_std':float(values.std()),
            f'{prefix}_p50':float(np.percentile(values,50)),f'{prefix}_p75':float(np.percentile(values,75)),
            f'{prefix}_p90':float(np.percentile(values,90)),f'{prefix}_p95':float(np.percentile(values,95)),
            f'{prefix}_max':float(values.max()),f'{prefix}_sum':float(values.sum())}

def extract_features(volume):
    loc = locate_striatum(volume)
    diagnostic_keys=('z_peak','midline_x','pair_half_distance','left_x','left_y','right_x','right_y','peak_distance','peak_y_difference','left_contrast','right_contrast','bilateral_score','score_prominence','boundary_distance','y_boundary_distance','localization_warning')
    features={k:loc[k] for k in diagnostic_keys}
    for radius in ROI_RADII:
        for half in SLAB_HALF_WIDTHS:
            left_values = sphere_values(volume,loc['left'],radius,half)
            right_values = sphere_values(volume,loc['right'],radius,half)
            lp, rp = f'roi_l_r{radius}_s{half}', f'roi_r_r{radius}_s{half}'
            features.update(summarize_values(left_values,lp))
            features.update(summarize_values(right_values,rp))
            for statistic in ('mean','p75','p90','p95','max','sum'):
                lv,rv = features[f'{lp}_{statistic}'],features[f'{rp}_{statistic}']
                features[f'asym_r{radius}_s{half}_{statistic}_absdiff'] = abs(lv-rv)
                features[f'asym_r{radius}_s{half}_{statistic}_reldiff'] = abs(lv-rv)/(abs(lv)+abs(rv)+1e-6)
                features[f'asym_r{radius}_s{half}_{statistic}_ratio'] = min(lv,rv)/(max(lv,rv)+1e-6)
    return features,loc

## 4. Auditoría visual estratificada por protocolo

In [ ]:
qc_sample = manifest.groupby('protocol_cluster',group_keys=False).sample(n=SAMPLES_PER_CLUSTER,random_state=RANDOM_STATE)
fig,axes = plt.subplots(len(qc_sample),2,figsize=(10,3*len(qc_sample)))
qc_rows=[]
for i,(_,row) in enumerate(qc_sample.reset_index(drop=True).iterrows()):
    volume=load_volume(row['processed_path'])
    _,loc=extract_features(volume)
    ax=axes[i,0]
    ax.imshow(loc['slab'].T,cmap='inferno',origin='lower',vmin=0,vmax=1)
    for center,color in ((loc['left'],'cyan'),(loc['right'],'lime')):
        ax.add_patch(Circle((center[0],center[1]),7,fill=False,color=color,linewidth=1.5))
    ax.set_title(f"C{row['protocol_cluster']} | {row['uid']} | z={loc['z_peak']}")
    ax.set_xlim(8,88);ax.set_ylim(8,88);ax.axis('off')
    axes[i,1].plot(range(SEARCH_Z[0],SEARCH_Z[1]),loc['z_profile'])
    axes[i,1].axvline(loc['z_peak'],color='crimson',linestyle='--')
    if loc['localization_warning']:
        ax.set_title(ax.get_title()+' | REVISAR',color='crimson')
    axes[i,1].set(title='Perfil de contraste bilateral',xlabel='z',ylabel='puntuación')
    qc_rows.append({'uid':row['uid'],'target':row['target'],'protocol_cluster':row['protocol_cluster'],
                    **{k:loc[k] for k in ('z_peak','midline_x','pair_half_distance','left_x','left_y','right_x','right_y','peak_distance','peak_y_difference','left_contrast','right_contrast','bilateral_score','score_prominence','boundary_distance','y_boundary_distance','localization_warning')}})
plt.tight_layout();plt.savefig(FEATURE_DIR/'roi_localization_qc.png',dpi=170,bbox_inches='tight');plt.show()
qc_df=pd.DataFrame(qc_rows);qc_df.to_csv(FEATURE_DIR/'roi_localization_qc.csv',index=False)
display(qc_df)

## 5. Aprobación manual
Los dos círculos deben cubrir las captaciones estriatales, no corteza, fondo ni captaciones periféricas. No continúe si fallan varios casos de un protocolo.

In [ ]:
ROI_AUDIT_APPROVED=False
print('Auditoría ROI aprobada:',ROI_AUDIT_APPROVED)

## 6. Extracción completa con punto de control

In [ ]:
if not ROI_AUDIT_APPROVED:
    raise RuntimeError('Revise la localización y cambie ROI_AUDIT_APPROVED=True.')
rows,errors=[],[]
start=time.time()
for i,row in manifest.iterrows():
    try:
        volume=load_volume(row['processed_path'])
        features,_=extract_features(volume)
        rows.append({'uid':row['uid'],'target':int(row['target']),'protocol_cluster':int(row['protocol_cluster']),'fold':int(row['fold']),**features})
    except Exception as exc:
        errors.append({'uid':row['uid'],'error_type':type(exc).__name__,'error':str(exc)})
    if (i+1)%50==0 or i+1==len(manifest):
        print(f"{i+1}/{len(manifest)} | correctos={len(rows)} | errores={len(errors)} | {(time.time()-start)/60:.1f} min")
        pd.DataFrame(rows).to_csv(FEATURE_DIR/'roi_features_checkpoint.csv',index=False)
features_df=pd.DataFrame(rows)
errors_df=pd.DataFrame(errors)
features_df.to_csv(FEATURE_DIR/'roi_features.csv',index=False)
errors_df.to_csv(FEATURE_DIR/'roi_feature_errors.csv',index=False)
print('Características:',features_df.shape,'| errores:',len(errors_df))

## 7. Auditoría numérica de la localización

In [ ]:
location_cols=['z_peak','midline_x','pair_half_distance','left_x','left_y','right_x','right_y','peak_distance','left_contrast','right_contrast','bilateral_score','score_prominence','boundary_distance','y_boundary_distance']
display(features_df.groupby('protocol_cluster')[location_cols].agg(['median','min','max']))
fig,axes=plt.subplots(1,3,figsize=(15,4))
sns.boxplot(data=features_df,x='protocol_cluster',y='z_peak',ax=axes[0])
sns.boxplot(data=features_df,x='protocol_cluster',y='peak_distance',ax=axes[1])
sns.boxplot(data=features_df,x='protocol_cluster',y='peak_y_difference',ax=axes[2])
plt.tight_layout();plt.savefig(FEATURE_DIR/'localization_by_protocol.png',dpi=170);plt.show()
warning_summary=features_df.groupby('protocol_cluster')['localization_warning'].agg(['sum','mean'])
display(warning_summary.rename(columns={'sum':'sospechosos','mean':'proporción'}))
print('UID completos:',set(features_df.uid)==set(manifest.uid),'| errores:',len(errors_df))

## 8. Elastic Net con hiperparámetros fijados
Se usa exactamente el mismo modelo en ambas CV. Los hiperparámetros no se ajustan con los folds externos, evitando optimismo. Solo entran variables con prefijo `roi_` o `asym_`.

In [ ]:
feature_columns=[c for c in features_df.columns if c.startswith('roi_') or c.startswith('asym_')]
X=features_df[feature_columns]
y=features_df['target'].astype(int).to_numpy()
print('Predictores:',len(feature_columns))

def make_model():
    return Pipeline([
        ('imputer',SimpleImputer(strategy='median',add_indicator=True)),
        ('variance',VarianceThreshold(1e-10)),
        ('scaler',StandardScaler()),
        ('model',LogisticRegression(penalty='elasticnet',solver='saga',C=0.05,l1_ratio=0.20,max_iter=6000,random_state=RANDOM_STATE))
    ])

def evaluate_splits(name,splits):
    oof=np.full(len(features_df),np.nan)
    fold_rows=[]
    for fold,(train_idx,valid_idx) in enumerate(splits):
        model=make_model();model.fit(X.iloc[train_idx],y[train_idx])
        pred=np.clip(model.predict_proba(X.iloc[valid_idx])[:,1],1e-6,1-1e-6)
        oof[valid_idx]=pred
        fold_rows.append({'scheme':name,'fold':fold,'n_train':len(train_idx),'n_valid':len(valid_idx),
                          'logloss':log_loss(y[valid_idx],pred),'auc':roc_auc_score(y[valid_idx],pred),
                          'brier':brier_score_loss(y[valid_idx],pred),
                          'valid_prevalence':float(y[valid_idx].mean())})
    if np.isnan(oof).any():raise RuntimeError('OOF incompleto.')
    overall={'scheme':name,'n':len(y),'logloss_oof':log_loss(y,oof),'auc_oof':roc_auc_score(y,oof),'brier_oof':brier_score_loss(y,oof)}
    return oof,pd.DataFrame(fold_rows),overall

skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
stratified_splits=list(skf.split(X,y))
grouped_splits=[]
for fold in sorted(features_df['fold'].unique()):
    valid_idx=np.flatnonzero(features_df['fold'].to_numpy()==fold)
    train_idx=np.flatnonzero(features_df['fold'].to_numpy()!=fold)
    grouped_splits.append((train_idx,valid_idx))

oof_strat,folds_strat,overall_strat=evaluate_splits('stratified',stratified_splits)
oof_group,folds_group,overall_group=evaluate_splits('protocol_grouped',grouped_splits)
results=pd.DataFrame([overall_strat,overall_group])
display(results)
display(pd.concat([folds_strat,folds_group],ignore_index=True))

## 9. Guardar resultados y modelo final exploratorio

In [ ]:
oof=pd.DataFrame({'uid':features_df.uid,'target':y,'protocol_cluster':features_df.protocol_cluster,'fold':features_df.fold,
                  'pred_stratified':oof_strat,'pred_protocol_grouped':oof_group})
oof.to_csv(FEATURE_DIR/'elasticnet_oof.csv',index=False)
pd.concat([folds_strat,folds_group],ignore_index=True).to_csv(FEATURE_DIR/'elasticnet_fold_metrics.csv',index=False)
results.to_csv(FEATURE_DIR/'elasticnet_overall_metrics.csv',index=False)
final_model=make_model();final_model.fit(X,y)
joblib.dump({'model':final_model,'feature_columns':feature_columns},FEATURE_DIR/'elasticnet_roi_final.joblib')
results['delta_vs_stratified']=results['logloss_oof']-results.loc[results.scheme=='stratified','logloss_oof'].iloc[0]
display(results)
print('Archivos guardados en:',FEATURE_DIR)